# **Building GenAI based Application Lab(MCSE641P)**
# **Assignment 7 - Demonstrate PEFT & LORA using OLLAMA**

Name: K.B.Girish [24MCS1050]

**Aim**: To demonstrate the working mechanism of PEFT & LORA using OLLAMA

## Import Required packages

In [ ]:
import os
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)
from peft import LoraConfig, get_peft_model, PeftModel, prepare_model_for_kbit_training

In [ ]:
# Enable deterministic behavior for reproducibility
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
# Set model name - using Phi-2 which is OLLAMA-compatible and very lightweight
model_name = "microsoft/phi-2"  # Only 2.7B parameters & OLLAMA compatible

In [ ]:
# Configure device - use CPU if GPU not available
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


### 1. Load the model and tokenizer

In [ ]:
# 1. Load the model and tokenizer
print("Loading base model and tokenizer...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    load_in_8bit=True if device == "cuda" else False,  # 8-bit only on CUDA
    device_map="auto" if device == "cuda" else None,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
)

Loading base model and tokenizer...


config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

c:\Users\Girish\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Girish\.cache\huggingface\hub\models--microsoft--phi-2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `Bit

model.safetensors.index.json:   0%|          | 0.00/35.7k [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/564M [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
# Prepare model for training if using 8-bit quantization
if device == "cuda":
    model = prepare_model_for_kbit_training(model)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

tokenizer_config.json:   0%|          | 0.00/7.34k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/1.08k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

### 2. Configure LoRA

In [ ]:
# 2. Configure LoRA
print("Configuring LoRA...")
lora_config = LoraConfig(
    r=4,                     # Very low rank dimension for memory efficiency
    lora_alpha=16,           # LoRA scaling factor
    target_modules=["q_proj", "k_proj", "v_proj"],  # Phi-2 specific attention module
    lora_dropout=0.05,       # Dropout for LoRA layers
    bias="none",             # Don't train bias terms
    task_type="CAUSAL_LM"    # Set task type for causal language modeling
)

Configuring LoRA...


### 3. Apply LoRA to the model

In [ ]:
# 3. Apply LoRA to the model
print("Applying LoRA adapters to the model...")
peft_model = get_peft_model(model, lora_config)

Applying LoRA adapters to the model...


### 4. Print trainable parameters to demonstrate parameter efficiency

In [ ]:
# 4. Print trainable parameters to demonstrate parameter efficiency
print("\n=== Parameter Efficiency Analysis ===")
total_params = sum(p.numel() for p in peft_model.parameters())
trainable_params = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Percentage of trainable parameters: {100 * trainable_params / total_params:.6f}%")


=== Parameter Efficiency Analysis ===
Total parameters: 2,781,649,920
Trainable parameters: 1,966,080
Percentage of trainable parameters: 0.070680%


### 5. Prepare a very small dataset for quick demonstration

In [ ]:
# 5. Prepare a very small dataset for quick demonstration (using a tiny subset of IMDB)
print("Preparing dataset...")
dataset = load_dataset("imdb", split="train[:40]")  # Just 40 examples to keep it super light

Preparing dataset...


### 6. Function to format data for instruction tuning - adapted for Phi-2

In [ ]:
# 6. Function to format data for instruction tuning - adapted for Phi-2
def format_instruction(text):
    """Format the dataset for instruction fine-tuning"""
    instruction = "Summarize the movie review in one sentence."
    # Phi-2 uses a simple instruction format
    return f"Instruction: {instruction}\nInput: {text}\nOutput:"

### 7. Tokenize and process the dataset

In [ ]:
# 7. Tokenize the dataset
def tokenize_function(examples):
    texts = examples["text"]
    formatted_texts = [format_instruction(text) for text in texts]

    return tokenizer(
        formatted_texts,
        padding="max_length",
        truncation=True,
        max_length=256,  # Short sequences for faster training
        return_tensors="np"  # Use numpy for CPU compatibility
    )

In [ ]:
# Process the dataset
tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    batch_size=8,  # Process in small batches
    remove_columns=["text", "label"]
)

Map:   0%|          | 0/40 [00:00<?, ? examples/s]

### 8. Data collator for language modeling

In [ ]:
# 8. Data collator for language modeling
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False  # Not using masked language modeling
)

### 9. Set up training arguments for minimal resource usage

In [ ]:
# 9. Set up training arguments for minimal resource usage
training_args = TrainingArguments(
    output_dir="./phi_lora_summary",
    per_device_train_batch_size=1,  # Tiny batch size for memory efficiency
    gradient_accumulation_steps=4,  # Accumulate gradients
    learning_rate=2e-4,
    num_train_epochs=1,             # Just 1 epoch for quick demonstration
    logging_steps=5,
    save_strategy="no",             # Don't save checkpoints to save time and space
    report_to="none",               # Disable wandb/tensorboard reporting
    fp16=True if device == "cuda" else False,  # Mixed precision only on GPU
    optim="adamw_torch",
    max_steps=15,                   # Limit to just 15 steps for a quick demo
)

### 10. Create trainer

In [ ]:
# 10. Create trainer
trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

### 11. Train the model

In [ ]:
# 11. Train the model
print("\n=== Starting LoRA fine-tuning ===")
print("This will be a minimal training run for demonstration purposes")
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.



=== Starting LoRA fine-tuning ===
This will be a minimal training run for demonstration purposes


c:\Users\Girish\anaconda3\Lib\site-packages\torch\_dynamo\eval_frame.py:632: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
c:\Users\Girish\anaconda3\Lib\site-packages\bitsandbytes\autograd\_functions.py:315: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss
5,3.205900
10,3.035600
15,3.031300


TrainOutput(global_step=15, training_loss=3.090948994954427, metrics={'train_runtime': 51.6887, 'train_samples_per_second': 1.161, 'train_steps_per_second': 0.29, 'total_flos': 244277261107200.0, 'train_loss': 3.090948994954427, 'epoch': 1.5})

### 12. Save the LoRA adapter weights

In [ ]:
# 12. Save the LoRA adapter weights (very small file)
print("\n=== Saving LoRA adapter weights ===")
peft_model.save_pretrained("./phi_lora_adapters")
print(f"LoRA adapters saved to ./phi_lora_adapters")
adapter_size = sum(os.path.getsize(os.path.join("./phi_lora_adapters", f)) for f in os.listdir("./phi_lora_adapters") if os.path.isfile(os.path.join("./phi_lora_adapters", f)))
print(f"Adapter size: {adapter_size / 1024 / 1024:.2f} MB")


=== Saving LoRA adapter weights ===
LoRA adapters saved to ./phi_lora_adapters
Adapter size: 7.53 MB


### 13. Function to generate summaries with the trained model

In [ ]:
# 13. Function to generate summaries with the trained model
def generate_summary(review_text):
    prompt = f"Instruction: Summarize the movie review in one sentence.\nInput: {review_text}\nOutput:"
    inputs = tokenizer(prompt, return_tensors="pt")

    # Move inputs to the right device
    if device == "cuda":
        inputs = {k: v.to("cuda") for k, v in inputs.items()}

    with torch.no_grad():
        output = peft_model.generate(
            inputs["input_ids"],
            max_new_tokens=40,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)

    # Extract just the summary part (after the prompt)
    summary = generated_text[len(prompt):].strip()
    return summary

### 14. Test with example reviews

In [ ]:
# 14. Test with example reviews
test_reviews = [
    "This movie was fantastic! The acting was superb and the plot kept me on the edge of my seat.",
    "I was disappointed by this film. The characters were poorly developed and the story didn't make sense.",
    "A masterpiece of modern cinema with breathtaking visuals and compelling performances."
]

print("\n=== Testing the LoRA-tuned model ===")
for i, review in enumerate(test_reviews):
    print(f"\nTest Review #{i+1}: {review}")
    summary = generate_summary(review)
    print(f"Generated Summary: {summary}")

c:\Users\Girish\anaconda3\Lib\site-packages\transformers\generation\configuration_utils.py:628: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\Girish\anaconda3\Lib\site-packages\transformers\generation\configuration_utils.py:633: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



=== Testing the LoRA-tuned model ===

Test Review #1: This movie was fantastic! The acting was superb and the plot kept me on the edge of my seat.


c:\Users\Girish\anaconda3\Lib\site-packages\torch\utils\checkpoint.py:87: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(


Generated Summary: The movie was a superb and suspenseful experience.

Test Review #2: I was disappointed by this film. The characters were poorly developed and the story didn't make sense.
Generated Summary: The film was a waste of time and money.

Test Review #3: A masterpiece of modern cinema with breathtaking visuals and compelling performances.
Generated Summary: A boring and pretentious waste of time and money.


### 15. Sample to create OLLAMA Modelfile for PHI-2

In [ ]:
# 15. Create OLLAMA Modelfile for PHI-2
print("\n=== Creating Modelfile for OLLAMA ===")
with open("Modelfile", "w") as f:
    f.write("""FROM phi

# Model parameters
PARAMETER temperature 0.7
PARAMETER top_k 40
PARAMETER top_p 0.9

# System prompt
SYSTEM \"\"\"
You are a helpful assistant that specializes in summarizing movie reviews in one sentence.
\"\"\"

# Template format
TEMPLATE \"\"\"
{{ .System }}

USER: {{ .Prompt }}
ASSISTANT:
\"\"\"
""")


=== Creating Modelfile for OLLAMA ===


### Results

In [ ]:
print("""
=== PEFT & LoRA Demonstration Complete ===

This script demonstrates:
1. Parameter-Efficient Fine-Tuning (PEFT) using LoRA with Phi-2 (OLLAMA-compatible model)
2. How LoRA dramatically reduces trainable parameters (as shown in Parameter Efficiency Analysis)
3. Fine-tuned model generating summaries from the learned task

For OLLAMA integration:
1. Trained with Phi-2 which is directly compatible with OLLAMA's "phi" model
2. Created a Modelfile compatible with OLLAMA's expectation format
3. Saved minimal LoRA adapter weights

To use with OLLAMA directly:
1. Create your model with: ollama create summary-phi -f Modelfile
2. Run with: ollama run summary-phi "This movie was amazing, with great performances!"

LoRA Technical Explanation:
Original weight matrix W frozen during training
Low-rank matrices A (d×r) and B (r×k) are trained where r << min(d,k)
During inference: W' = W + A·B (merged weight matrix)
This allows fine-tuning on consumer hardware with minimal parameters
""")


=== PEFT & LoRA Demonstration Complete ===

This script demonstrates:
1. Parameter-Efficient Fine-Tuning (PEFT) using LoRA with Phi-2 (OLLAMA-compatible model)
2. How LoRA dramatically reduces trainable parameters (as shown in Parameter Efficiency Analysis)
3. Fine-tuned model generating summaries from the learned task

For OLLAMA integration:
1. Trained with Phi-2 which is directly compatible with OLLAMA's "phi" model
2. Created a Modelfile compatible with OLLAMA's expectation format
3. Saved minimal LoRA adapter weights

To use with OLLAMA directly:
1. Create your model with: ollama create summary-phi -f Modelfile
2. Run with: ollama run summary-phi "This movie was amazing, with great performances!"

LoRA Technical Explanation:
Original weight matrix W frozen during training
Low-rank matrices A (d×r) and B (r×k) are trained where r << min(d,k)
During inference: W' = W + A·B (merged weight matrix)
This allows fine-tuning on consumer hardware with minimal parameters



### Show model size comparison

In [ ]:
model_size = sum(p.numel() * p.element_size() for p in model.parameters()) / (1024**3)
adapter_size_gb = adapter_size / (1024**3)
print(f"\nModel Size Comparison:")
print(f"Full model size: {model_size:.2f} GB")
print(f"LoRA adapter size: {adapter_size_gb:.6f} GB")
print(f"Size reduction: {model_size/adapter_size_gb:.1f}x smaller")


Model Size Comparison:
Full model size: 3.33 GB
LoRA adapter size: 0.007353 GB
Size reduction: 453.0x smaller
